In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

def build_gold_features_cliente():
    print("Construindo a view gold_features_cliente para Data Science...")
    
    query = """
    CREATE OR REPLACE VIEW workspace.default.gold_features_cliente AS
    
    SELECT 
        c.id_cliente,
        c.renda,
        c.segmento,
        
        -- Features Comportamentais (trazendo da gold_cliente_mes)
        COALESCE(m.gasto_total_mes, 0) AS ultimo_gasto_mensal,
        COALESCE(m.qtd_transacoes_mes, 0) AS freq_transacoes_mensal,
        
        -- Features de Risco (trazendo da gold_indicadores_risco)
        COALESCE(r.qtd_eventos_risco, 0) AS historico_eventos_risco,
        COALESCE(r.qtd_estornos, 0) AS historico_estornos
        
    FROM workspace.default.silver_clientes c
    
    -- Pega apenas o comportamento do mês atual/mais recente
    LEFT JOIN workspace.default.gold_cliente_mes m 
        ON c.id_cliente = m.id_cliente 
        -- Pega o mês máximo gerado na base
        AND m.mes_referencia = (SELECT MAX(mes_referencia) FROM workspace.default.gold_cliente_mes)
        
    LEFT JOIN workspace.default.gold_indicadores_risco r 
        ON c.id_cliente = r.id_cliente
        
    -- Regra fundamental para DS: Apenas clientes com cadastro vigente (SCD2 = Ativo)
    WHERE c.is_active = TRUE;
    """
    
    spark.sql(query)
    print("Sucesso! View workspace.default.gold_features_cliente criada.")

build_gold_features_cliente()